# Mental Health Vulnerability Across Age Groups in Australia
## Psychological Distress, Contributing Factors and Service Access

**Author:** Robin Anand  
**Course:** DATA7001 - Introduction to Data Science, University of Queensland  

---

### Data Sources
- ABS National Study of Mental Health and Wellbeing 2020-2022
- AIHW Mental Health Services in Australia 2022-2023
- Mission Australia Youth Mental Health Survey 2012-2023
- ABS National Health Survey 2022

### Research Questions
1. Which age group in Australia experiences the highest levels of psychological distress?
2. How has distress changed over time across age groups?
3. What factors most strongly contribute to mental health vulnerability?
4. Is there a gap between distress rates and service access across age groups?

### Project Structure
```
australian-mental-health-analysis/
├── README.md
├── australian_mental_health_analysis.png
├── src/
│   └── australian_mental_health_analysis.ipynb
└── data/
    ├── distress_by_age.csv
    ├── trend.csv
    ├── factors.csv
    └── services.csv
```

In [ ]:
# Import libraries
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

print('Libraries imported successfully')

## 1. Load Data from CSV Files
Note: CSV files are located in the `../data/` folder relative to this notebook in `src/`

In [ ]:
# Load all datasets from the data folder
# '../data/' means go up one level from src/ then into data/
distress_by_age = pd.read_csv('../data/distress_by_age.csv')
trend           = pd.read_csv('../data/trend.csv')
factors         = pd.read_csv('../data/factors.csv')
services        = pd.read_csv('../data/services.csv')

print('All datasets loaded successfully')
print(f'distress_by_age: {distress_by_age.shape[0]} rows, {distress_by_age.shape[1]} columns')
print(f'trend:           {trend.shape[0]} rows, {trend.shape[1]} columns')
print(f'factors:         {factors.shape[0]} rows, {factors.shape[1]} columns')
print(f'services:        {services.shape[0]} rows, {services.shape[1]} columns')

## 2. Preview the Data

In [ ]:
print('--- Distress by Age Group ---')
print(distress_by_age)
print()
print('--- Trend Over Time ---')
print(trend)
print('--- Contributing Factors ---')
print(factors)
print()
print('--- Service Access vs Distress ---')
print(services)

## 3. Exploratory Data Analysis

In [ ]:
# Key statistics
most_vulnerable  = distress_by_age.loc[distress_by_age.high_very_high_pct.idxmax()]
least_vulnerable = distress_by_age.loc[distress_by_age.high_very_high_pct.idxmin()]
youth_2012       = trend[trend.year == 2012].youth.values[0]
youth_2022       = trend[trend.year == 2022].youth.values[0]
strongest_factor = factors.loc[factors.odds_ratio.idxmax()]

print('=== KEY FINDINGS ===')
print(f'Most vulnerable age group:  {most_vulnerable.age_group} ({most_vulnerable.high_very_high_pct}%)')
print(f'Least vulnerable age group: {least_vulnerable.age_group} ({least_vulnerable.high_very_high_pct}%)')
print(f'Youth distress 2012:        {youth_2012}%')
print(f'Youth distress 2022:        {youth_2022}%')
print(f'Increase over 10 years:     {youth_2022 - youth_2012:.1f}%')
print(f'Strongest risk factor:      {strongest_factor.factor} (OR={strongest_factor.odds_ratio})')
print(f'Gender gap (16-24):         Female {distress_by_age.iloc[0].female_pct}% vs Male {distress_by_age.iloc[0].male_pct}%')


# Descriptive statistics
print('=== DESCRIPTIVE STATISTICS: Distress by Age Group ===')
print(distress_by_age.describe())

## 4. Visualisation - 4 Chart Dashboard

In [ ]:
# Sort factors for horizontal bar chart
factors_sorted = factors.sort_values('odds_ratio', ascending=True)

# Colour palette
BLUE   = '#2E75B6'
TEAL   = '#1F7A6E'
CORAL  = '#C0392B'
AMBER  = '#E67E22'
PURPLE = '#6C3483'
GRAY   = '#7F8C8D'

plt.rcParams.update({
    'font.family':       'DejaVu Sans',
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'axes.grid':         True,
    'grid.alpha':        0.3,
    'grid.linestyle':    '--'
})

fig = plt.figure(figsize=(16, 12))
fig.patch.set_facecolor('#FAFAFA')
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.42, wspace=0.32)

# Chart 1: Distress by Age Group
ax1   = fig.add_subplot(gs[0, 0])
x     = np.arange(len(distress_by_age['age_group']))
width = 0.28
ax1.bar(x - width, distress_by_age['high_very_high_pct'], width, label='Overall', color=BLUE,  alpha=0.88)
ax1.bar(x,          distress_by_age['female_pct'],         width, label='Female',  color=CORAL, alpha=0.88)
ax1.bar(x + width,  distress_by_age['male_pct'],           width, label='Male',    color=TEAL,  alpha=0.88)
ax1.set_xticks(x)
ax1.set_xticklabels(distress_by_age['age_group'], fontsize=9)
ax1.set_ylabel('Population with High/Very High Distress (%)', fontsize=9)
ax1.set_title('Psychological Distress by Age Group & Gender\n(ABS NSMHW 2020-2022)', fontsize=10, fontweight='bold', pad=10)
ax1.legend(fontsize=8, framealpha=0.5)
ax1.set_ylim(0, 42)
for bar in ax1.patches:
    h = bar.get_height()
    if h > 0:
        ax1.text(bar.get_x() + bar.get_width()/2, h+0.5, f'{h:.1f}%', ha='center', va='bottom', fontsize=7)

# Chart 2: Trend Over Time
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(trend['year'], trend['youth'],   marker='o', color=CORAL, linewidth=2.2, label='Youth (15-24)',  markersize=6)
ax2.plot(trend['year'], trend['adults'],  marker='s', color=BLUE,  linewidth=2.2, label='Adults (25-64)', markersize=6)
ax2.plot(trend['year'], trend['elderly'], marker='^', color=TEAL,  linewidth=2.2, label='Elderly (65+)',  markersize=6)
ax2.fill_between(trend['year'], trend['youth'], trend['elderly'], alpha=0.07, color=CORAL)
ax2.set_xticks(trend['year'])
ax2.set_ylabel('High/Very High Distress (%)', fontsize=9)
ax2.set_title('Psychological Distress Trends by Age Group\n(2012-2023)', fontsize=10, fontweight='bold', pad=10)
ax2.legend(fontsize=8, framealpha=0.5)
ax2.set_ylim(0, 36)
for year, youth in zip(trend['year'], trend['youth']):
    ax2.annotate(f'{youth}%', (year, youth), textcoords='offset points', xytext=(0, 6), ha='center', fontsize=7, color=CORAL)

# Chart 3: Contributing Factors
ax3    = fig.add_subplot(gs[1, 0])
colors = [CORAL if v >= 3.0 else AMBER if v >= 2.0 else GRAY for v in factors_sorted['odds_ratio']]
bars   = ax3.barh(factors_sorted['factor'], factors_sorted['odds_ratio'], color=colors, alpha=0.88, height=0.6)
ax3.axvline(x=1, color='black', linewidth=1, linestyle='--', alpha=0.5)
ax3.set_xlabel('Odds Ratio - Higher = Stronger Risk', fontsize=9)
ax3.set_title('Contributing Factors to Psychological Distress\n(ABS National Health Survey 2022)', fontsize=10, fontweight='bold', pad=10)
for bar, val in zip(bars, factors_sorted['odds_ratio']):
    ax3.text(val+0.05, bar.get_y()+bar.get_height()/2, f'OR = {val:.1f}', va='center', fontsize=8, fontweight='bold')
ax3.set_xlim(0, 4.6)
from matplotlib.patches import Patch
ax3.legend(handles=[
    Patch(facecolor=CORAL, label='High risk (OR >= 3.0)'),
    Patch(facecolor=AMBER, label='Moderate risk (OR >= 2.0)'),
    Patch(facecolor=GRAY,  label='Lower risk (OR < 2.0)')
], fontsize=7.5, framealpha=0.5, loc='lower right')

# Chart 4: Distress vs Service Access
ax4 = fig.add_subplot(gs[1, 1])
ax4.scatter(services['distress_rate'], services['service_access'],
            s=180, c=[CORAL,BLUE,TEAL,AMBER,PURPLE,GRAY],
            alpha=0.88, zorder=5, edgecolors='white', linewidths=1.5)
for _, row in services.iterrows():
    ax4.annotate(row['age_group'], (row['distress_rate'], row['service_access']),
                 textcoords='offset points', xytext=(6, 4), fontsize=8.5, fontweight='bold')
z      = np.polyfit(services['distress_rate'], services['service_access'], 1)
p      = np.poly1d(z)
x_line = np.linspace(services['distress_rate'].min()-1, services['distress_rate'].max()+1, 100)
ax4.plot(x_line, p(x_line), '--', color=GRAY, alpha=0.6, linewidth=1.5, label='Trend line')
ax4.set_xlabel('Population with High/Very High Distress (%)', fontsize=9)
ax4.set_ylabel('Mental Health Service Access Rate (%)', fontsize=9)
ax4.set_title('Mental Health Distress vs Service Access by Age Group\n(AIHW 2022-2023)', fontsize=10, fontweight='bold', pad=10)
ax4.legend(fontsize=8, framealpha=0.5)

fig.suptitle(
    'Mental Health Vulnerability Across Age Groups in Australia\n'
    'Psychological Distress, Contributing Factors and Service Access',
    fontsize=14, fontweight='bold', y=0.98, color='#1A1A2E'
)
fig.text(0.5, 0.01,
    'Sources: ABS NSMHW 2020-2022 | AIHW Mental Health Services 2022-2023 | Mission Australia Youth Survey 2023',
    ha='center', fontsize=7.5, color=GRAY, style='italic')

plt.savefig('../australian_mental_health_analysis.png', dpi=150, bbox_inches='tight', facecolor='#FAFAFA')
plt.show()
print('Chart saved successfully')

## 5. Key Findings

### RQ1 - Most Vulnerable Age Group
Young adults aged **16-24** experience the highest rates of psychological distress at **25.7%** overall, with females in this group reaching **34.2%**.

### RQ2 - Trends Over Time
Youth distress increased from **18.7% in 2012 to 28.8% in 2022** — a rise of 10.1 percentage points over a decade.

### RQ3 - Contributing Factors
Financial stress (OR=3.8) and unemployment (OR=3.2) are the strongest risk factors, followed by social isolation (OR=2.9).

### RQ4 - Service Access Gap
Despite having the highest distress rates, young adults aged 16-24 have the lowest service access (38.2%), suggesting a significant service gap for the most vulnerable group.

## 6. Implications for Stakeholders
- **Policymakers** should prioritise funding for youth mental health services
- **Mental health services** should focus on financial stress and unemployment as modifiable risk factors
- **Schools and universities** should implement early intervention programs for the 16-24 age group
- **Employers** should address financial stress and job insecurity as primary workplace mental health concerns